# Renombrado de hojas Excel con longitudes de onda

Este notebook renombra las hojas de banda en los 7 archivos Excel hiperespectrales,
agregando la longitud de onda central de cada banda al nombre de la hoja.

**Transformación:** `B000` → `B000 399.87nm`

**Estrategia:** Manipulación directa del ZIP interno del `.xlsx` (sin cargar celdas).
Solo se modifican `xl/workbook.xml` y `docProps/app.xml` — los únicos dos archivos
que contienen los nombres de hojas. El resto del archivo se copia intacto.
Esto evita cargar ~49 millones de celdas por archivo y reduce el tiempo de
procesamiento de horas a ~30 segundos por archivo.

## Celda 1 — Instalación de dependencias y montaje de Drive

In [1]:
!pip install spectral --quiet

from google.colab import drive
drive.mount('/content/drive')

print('✓ Dependencias listas y Drive montado.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.0/249.0 kB 5.8 MB/s eta 0:00:00
Mounted at /content/drive
✓ Dependencias listas y Drive montado.


## Celda 2 — Extracción de longitudes de onda desde el archivo .hdr

Las longitudes de onda son idénticas en los 7 cubos (diferencia máxima = 0.000000 nm),
por lo que basta con parsear un único archivo `.hdr` de referencia.

In [2]:
import spectral

# Ruta al archivo .hdr de referencia (HN2 como fuente canónica de wavelengths)
HDR_PATH = '/content/drive/MyDrive/Cubos2/HN2/raw_0_HN2.hdr'

img = spectral.open_image(HDR_PATH)
wavelengths = img.bands.centers  # lista de 325 floats en nm

assert len(wavelengths) == 325, (
    f'Se esperaban 325 longitudes de onda, se encontraron {len(wavelengths)}'
)

print(f'✓ {len(wavelengths)} longitudes de onda cargadas.')
print(f'   Rango: {wavelengths[0]:.2f}nm – {wavelengths[-1]:.2f}nm')
print(f'   Ejemplos: B000 → {wavelengths[0]:.2f}nm | B001 → {wavelengths[1]:.2f}nm | B324 → {wavelengths[324]:.2f}nm')

✓ 325 longitudes de onda cargadas.
   Rango: 399.87nm – 1000.39nm
   Ejemplos: B000 → 399.87nm | B001 → 401.72nm | B324 → 1000.39nm


## Celda 3 — Renombrado de hojas (loop principal)

Para cada archivo Excel:
1. Se abre el `.xlsx` como ZIP (es un ZIP de XMLs internamente)
2. Se leen los dos XMLs que contienen nombres de hojas y se aplica el reemplazo con regex
3. Se escribe el archivo de salida con los XMLs modificados y el resto intacto
4. La hoja `Metadatos` se ignora automáticamente (no coincide con el patrón `B\d{3}`)

Los originales **no se modifican**. Los archivos de salida se guardan en una nueva carpeta.

In [3]:
import os

path = '/content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos/'

if os.path.isdir(path):
    print("El directorio EXISTE")
    print("Archivos:", os.listdir(path))
else:
    print("El directorio NO existe")
    # Verificar qué parte del path falla
    parts = path.strip('/').split('/')
    for i in range(1, len(parts) + 1):
        sub = '/' + '/'.join(parts[:i])
        exists = os.path.isdir(sub)
        print(f"  {'✓' if exists else '✗'} {sub}")

El directorio NO existe
  ✓ /content
  ✓ /content/drive
  ✓ /content/drive/MyDrive
  ✓ /content/drive/MyDrive/Tesis Marcos
  ✓ /content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX
  ✗ /content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos


In [4]:
import os

parent = '/content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/'
print("Contenido:")
for item in os.listdir(parent):
    print(f"  {'[DIR]' if os.path.isdir(os.path.join(parent, item)) else '[FILE]'} '{item}'")

Contenido:
  [DIR] 'Cubos con Metadatos y Wavelength'


In [ ]:
import os

path = '/content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos y Wavelength/'
for f in os.listdir(path):
    print(f"  {f}")


In [5]:
import os

# La carpeta donde PUTOLUISDANIEL.ipynb los exportó originalmente
path = '/content/drive/MyDrive/Cubos_Exportados_XLSX/'
if os.path.isdir(path):
    for f in os.listdir(path):
        print(f"  {f}")
else:
    print("Tampoco existe")


  Cubo_HN2_normalizado.xlsx
  Cubo_JMM1_normalizado.xlsx
  Cubo_LFH2_normalizado.xlsx
  Cubo_LIMON1_3_normalizado.xlsx
  Cubo_LIMON2_4_normalizado.xlsx
  Cubo_TEJON_5_normalizado.xlsx
  Cubo_TEJON2_6_normalizado.xlsx
  Cubo_NAC2_normalizado.xlsx
  Cubo_UNAM_normalizado.xlsx
  Cubos_Metadatos


In [6]:
import zipfile
import re
import gc
import os
import time

# ── Rutas ─────────────────────────────────────────────────────────────────────
INPUT_DIR = '/content/drive/MyDrive/Cubos_Exportados_XLSX/Cubos_Metadatos'
OUTPUT_DIR = '/content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos y Wavelength/'

ARCHIVOS = [
    'Cubo_HN2_normalizado.xlsx',
    'Cubo_JMM1_normalizado.xlsx',
    'Cubo_LFH2_normalizado.xlsx',
    'Cubo_LIMON1_3_normalizado.xlsx',
    'Cubo_LIMON2_4_normalizado.xlsx',
    'Cubo_TEJON2_6_normalizado.xlsx',
    'Cubo_NAC2_normalizado.xlsx',
    'Cubo_UNAM_normalizado.xlsx',
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Funciones de modificación XML ─────────────────────────────────────────────

def _reemplazo_workbook(match, wavelengths):
    """Reemplaza name="B{nnn}" por name="B{nnn} {wavelength:.2f}nm"."""
    idx = int(match.group(1))
    if idx >= len(wavelengths):
        raise ValueError(f'Índice de banda {idx} fuera de rango (max {len(wavelengths) - 1})')
    return f'name="B{idx:03d} {wavelengths[idx]:.2f}nm"'

def _reemplazo_app(match, wavelengths):
    """Reemplaza <vt:lpstr>B{nnn}</vt:lpstr> por la versión con wavelength."""
    idx = int(match.group(1))
    if idx >= len(wavelengths):
        return match.group(0)  # fuera de rango: dejar intacto
    return f'<vt:lpstr>B{idx:03d} {wavelengths[idx]:.2f}nm</vt:lpstr>'

def modificar_xml(xml_bytes, patron, reemplazo_fn, wavelengths):
    """Aplica regex sobre el XML y devuelve los bytes modificados."""
    xml_str = xml_bytes.decode('utf-8')
    xml_modificado = re.sub(patron, lambda m: reemplazo_fn(m, wavelengths), xml_str)
    return xml_modificado.encode('utf-8')

# Patrones regex para los dos XMLs objetivo
PATRON_WORKBOOK = r'name="B(\d{3})"'
PATRON_APP      = r'<vt:lpstr>B(\d{3})</vt:lpstr>'

MODIFICADORES = {
    'xl/workbook.xml':  (PATRON_WORKBOOK, _reemplazo_workbook),
    'docProps/app.xml': (PATRON_APP,      _reemplazo_app),
}

def renombrar_hojas_xlsx(input_path, output_path, wavelengths):
    """
    Copia el archivo .xlsx al destino y renombra las hojas de banda
    modificando directamente los XMLs internos del ZIP.
    Devuelve el número de hojas renombradas.
    """
    hojas_renombradas = 0

    with zipfile.ZipFile(input_path, 'r') as zin:
        with zipfile.ZipFile(output_path, 'w', compression=zipfile.ZIP_DEFLATED) as zout:
            for item in zin.infolist():
                data = zin.read(item.filename)

                if item.filename in MODIFICADORES:
                    patron, reemplazo_fn = MODIFICADORES[item.filename]
                    antes = len(re.findall(patron, data.decode('utf-8')))
                    data = modificar_xml(data, patron, reemplazo_fn, wavelengths)
                    if item.filename == 'xl/workbook.xml':
                        hojas_renombradas = antes  # solo contamos desde workbook

                zout.writestr(item, data)

    return hojas_renombradas

# ── Loop principal ─────────────────────────────────────────────────────────────

errores = []

for nombre_archivo in ARCHIVOS:
    input_path  = os.path.join(INPUT_DIR,  nombre_archivo)
    output_path = os.path.join(OUTPUT_DIR, nombre_archivo)

    print(f"\n{'='*65}")
    print(f'Procesando: {nombre_archivo}')

    if not os.path.exists(input_path):
        msg = f'Archivo no encontrado: {input_path}'
        print(f'  ⚠ ERROR: {msg}')
        errores.append({'archivo': nombre_archivo, 'error': msg})
        continue

    try:
        t0 = time.time()
        n_renombradas = renombrar_hojas_xlsx(input_path, output_path, wavelengths)
        elapsed = time.time() - t0

        print(f'  ✓ {n_renombradas} hojas de banda renombradas en {elapsed:.1f}s')
        print(f'  → Guardado en: {output_path}')

    except Exception as exc:
        msg = str(exc)
        print(f'  ✗ ERROR: {msg}')
        errores.append({'archivo': nombre_archivo, 'error': msg})
        if os.path.exists(output_path):
            os.remove(output_path)  # limpiar archivo parcialmente escrito

    gc.collect()

print(f"\n{'='*65}")
print('Procesamiento completado.')
if errores:
    print(f'\n⚠ {len(errores)} archivo(s) con errores (ver detalle en Celda 4).')
else:
    print('✓ Todos los archivos procesados sin errores.')


Procesando: Cubo_HN2_normalizado.xlsx
  ✓ 325 hojas de banda renombradas en 55.0s
  → Guardado en: /content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos y Wavelength/Cubo_HN2_normalizado.xlsx

Procesando: Cubo_JMM1_normalizado.xlsx
  ✓ 325 hojas de banda renombradas en 48.7s
  → Guardado en: /content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos y Wavelength/Cubo_JMM1_normalizado.xlsx

Procesando: Cubo_LFH2_normalizado.xlsx
  ✓ 325 hojas de banda renombradas en 45.2s
  → Guardado en: /content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos y Wavelength/Cubo_LFH2_normalizado.xlsx

Procesando: Cubo_LIMON1_3_normalizado.xlsx
  ✓ 325 hojas de banda renombradas en 44.9s
  → Guardado en: /content/drive/MyDrive/Tesis Marcos/Cubos_Exportados_XLSX/Cubos con Metadatos y Wavelength/Cubo_LIMON1_3_normalizado.xlsx

Procesando: Cubo_LIMON2_4_normalizado.xlsx
  ✓ 325 hojas de banda renombradas en 43.5s
  → Guardado en: /content/driv

## Celda 4 — Verificación y reporte final

Para cada archivo generado exitosamente se confirma:
- Total de hojas (debe ser 326)
- Hoja `Metadatos` presente e intacta
- Primeras 3 y últimas 2 hojas de banda con el formato correcto

In [7]:
import xml.etree.ElementTree as ET

# Namespace del workbook.xml en formato OOXML
NS_SPREADSHEET = 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'

# Solo verificar archivos que se procesaron sin error
archivos_ok = [f for f in ARCHIVOS if not any(e['archivo'] == f for e in errores)]

for nombre_archivo in archivos_ok:
    output_path = os.path.join(OUTPUT_DIR, nombre_archivo)

    print(f"\n{'='*65}")
    print(f'Verificando: {nombre_archivo}')

    try:
        with zipfile.ZipFile(output_path, 'r') as z:
            xml_bytes = z.read('xl/workbook.xml')

        # Parsear la lista de hojas desde workbook.xml
        root = ET.fromstring(xml_bytes.decode('utf-8'))
        sheets_node = root.find(f'{{{NS_SPREADSHEET}}}sheets')
        hojas = [
            s.get('name')
            for s in sheets_node.findall(f'{{{NS_SPREADSHEET}}}sheet')
        ]

        total = len(hojas)
        ok_total = '✓' if total == 326 else '✗'
        print(f'  {ok_total} Total de hojas: {total} (esperado: 326)')

        # Verificar hoja Metadatos
        ok_meta = hojas[0] == 'Metadatos'
        print(f"  {'✓' if ok_meta else '✗'} Primera hoja: '{hojas[0]}' {'(Metadatos intacta)' if ok_meta else '(se esperaba Metadatos)'}")

        # Primeras 3 hojas de banda (posiciones 1, 2, 3)
        print('  Primeras 3 hojas de banda:')
        for h in hojas[1:4]:
            print(f'    {h}')

        # Últimas 2 hojas de banda
        print('  Últimas 2 hojas de banda:')
        for h in hojas[-2:]:
            print(f'    {h}')

    except Exception as exc:
        print(f'  ✗ Error al verificar: {exc}')

# ── Reporte final de errores ───────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f'RESUMEN FINAL')
print(f'  Archivos procesados correctamente: {len(archivos_ok)}/{len(ARCHIVOS)}')

if errores:
    print(f'\n⚠ Archivos con errores ({len(errores)}):')
    for e in errores:
        print(f"  - {e['archivo']}")
        print(f"    Error: {e['error']}")
else:
    print('  ✓ Sin errores. Todos los archivos están listos en:')
    print(f'    {OUTPUT_DIR}')


Verificando: Cubo_HN2_normalizado.xlsx
  ✓ Total de hojas: 326 (esperado: 326)
  ✓ Primera hoja: 'Metadatos' (Metadatos intacta)
  Primeras 3 hojas de banda:
    B000 399.87nm
    B001 401.72nm
    B002 403.57nm
  Últimas 2 hojas de banda:
    B323 998.54nm
    B324 1000.39nm

Verificando: Cubo_JMM1_normalizado.xlsx
  ✓ Total de hojas: 326 (esperado: 326)
  ✓ Primera hoja: 'Metadatos' (Metadatos intacta)
  Primeras 3 hojas de banda:
    B000 399.87nm
    B001 401.72nm
    B002 403.57nm
  Últimas 2 hojas de banda:
    B323 998.54nm
    B324 1000.39nm

Verificando: Cubo_LFH2_normalizado.xlsx
  ✓ Total de hojas: 326 (esperado: 326)
  ✓ Primera hoja: 'Metadatos' (Metadatos intacta)
  Primeras 3 hojas de banda:
    B000 399.87nm
    B001 401.72nm
    B002 403.57nm
  Últimas 2 hojas de banda:
    B323 998.54nm
    B324 1000.39nm

Verificando: Cubo_LIMON1_3_normalizado.xlsx
  ✓ Total de hojas: 326 (esperado: 326)
  ✓ Primera hoja: 'Metadatos' (Metadatos intacta)
  Primeras 3 hojas de banda:
